# 01_vectors_similarity: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/01_vectors_similarity/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Build exact nearest-neighbor search from hand features.

Concepts: 2D vectors, dot, cosine, Euclidean, normalization.
Run: uv run python modules/01_vectors_similarity/examples/01_from_scratch.py
Expected: cosine favors mug-warmer; after rescaling, dot favors travel-flask
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

import numpy as np

from common.teaching import hand_features
from tinyvec.exact import exact_knn


def main() -> None:
    ids, matrix, query = hand_features()
    print(f"Rows x coordinates: {matrix.shape}; dtype: {matrix.dtype}")
    print("Coordinates: warmth, portability; these are designed features, not learned meanings.")
    for metric in ["dot", "cosine", "l2"]:
        print(metric, exact_knn(matrix, query, ids, 4, metric))
    # KEY IDEA: a longer arrow can win a dot product even when its direction fits less well.
    scaled = matrix.copy()
    scaled[1] *= 3
    print("After multiplying the flask vector by 3:", exact_knn(scaled, query, ids, 1, "dot"))
    unit = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    print("Unit-vector lengths:", np.linalg.norm(unit, axis=1).round(4).tolist())


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/01_vectors_similarity/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Compare the exact kernel with native FAISS.

    Concepts: flat index, squared L2, score direction, neighbor agreement.
    Run: uv run --extra libs python modules/01_vectors_similarity/examples/02_production.py
    Expected: same L2 neighbor order; FAISS returns squared distance
    CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
    Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
    """

    # course-extra: libs
    import faiss
    import numpy as np

    from common.teaching import hand_features
    from tinyvec.exact import exact_knn

    def main() -> None:
        ids, matrix, query = hand_features()
        faiss.omp_set_num_threads(1)
        index = faiss.IndexFlatL2(matrix.shape[1])
        index.add(matrix)
        squared, positions = index.search(query[None, :], len(ids))
        expected = exact_knn(matrix, query, ids, len(ids), "l2")
        actual = [ids[int(i)] for i in positions[0]]
        assert actual == [key for key, _ in expected]
        # KEY IDEA: convert squared distance before comparing to TinyVec's negative L2 score.
        np.testing.assert_allclose(
            -np.sqrt(squared[0]), [score for _, score in expected], atol=1e-6
        )
        print("Agreement:", actual)
        print("FAISS squared distances:", squared[0].round(4).tolist())

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/01_vectors_similarity/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Observe distance concentration under a controlled distribution.

Concepts: random math experiment, distance spread, KDTree versus brute force.
Run: uv run python modules/01_vectors_similarity/examples/03_measure.py
Expected: relative spread generally shrinks; measured tree/brute times
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

import numpy as np
from sklearn.neighbors import NearestNeighbors
from threadpoolctl import threadpool_limits

from common.teaching import evidence, timed


def main() -> None:
    rng = np.random.default_rng(20261104)
    rows = []
    with threadpool_limits(limits=1):
        for dimensions in [2, 8, 32, 128, 512]:
            matrix = rng.normal(size=(1000, dimensions)).astype(np.float32)
            queries = rng.normal(size=(20, dimensions)).astype(np.float32)
            distances = np.linalg.norm(matrix - queries[0], axis=1)
            row = {
                "dimensions": dimensions,
                "coefficient_of_variation": float(distances.std() / distances.mean()),
            }
            for method in ["brute", "kd_tree"]:
                model = NearestNeighbors(algorithm=method, n_neighbors=5, n_jobs=1).fit(matrix)
                row[method] = timed(lambda model=model, queries=queries: model.kneighbors(queries))
            rows.append(row)
            print(
                dimensions,
                round(row["coefficient_of_variation"], 4),
                {m: round(row[m]["median_ms"], 3) for m in ["brute", "kd_tree"]},
            )
    # KEY IDEA: shrinking relative spread is distribution-dependent, not proof every real dataset is hopeless.
    assert rows[-1]["coefficient_of_variation"] < rows[0]["coefficient_of_variation"]
    evidence(
        "dimensions",
        {"distribution": "independent standard normal; math-only demonstration", "rows": rows},
    )


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 04_geometry

Source: `modules/01_vectors_similarity/examples/04_geometry.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Draw hand-designed 2D and 3D product coordinates before using formulas.

Concepts: a vector as a point/arrow, units, third coordinate, projection.
Run: uv run python modules/01_vectors_similarity/examples/04_geometry.py
Expected: geometry.png with labeled 2D/3D product positions under the course cache.
CPU: about 2 seconds; tiny hand features, no downloads, models, or Docker.
"""

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from common.settings import cache_dir
from common.teaching import hand_features


def main() -> None:
    ids, vectors, query = hand_features()
    # KEY IDEA: a third coordinate adds information that a 2D projection can hide.
    three = np.column_stack([vectors, [0.2, 0.1, 1.0, 0.4]])
    fig = plt.figure(figsize=(10, 4), layout="constrained")
    left = fig.add_subplot(121)
    right = fig.add_subplot(122, projection="3d")
    for key, point, point3 in zip(ids, vectors, three, strict=True):
        left.annotate(key, point, xytext=(4, 5), textcoords="offset points", fontsize=8)
        left.scatter(*point)
        right.scatter(*point3, label=key)
    left.arrow(0, 0, *query, width=0.006, color="black", length_includes_head=True)
    left.set(
        xlabel="Warmth (designed score)",
        ylabel="Portability (designed score)",
        xlim=(-0.1, 1.4),
        ylim=(-0.1, 1.2),
        title="Two coordinates and the query arrow",
    )
    right.set(
        xlabel="Warmth",
        ylabel="Portability",
        zlabel="Desk footprint",
        title="A third coordinate changes distance",
    )
    right.legend(fontsize=7)
    path = cache_dir() / "tier1" / "geometry.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(path)


if __name__ == "__main__":
    main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import hand_features
from tinyvec.exact import exact_knn

ids, matrix, query = hand_features()
# Change the query coordinates and predict which product wins.
query[:] = [0.2, 1.0]
exact_knn(matrix, query, ids, 3)

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.